# M01 — Fundamentos y entorno

[← Anterior](../M00-entorno-notebooks/02-lab-primer-notebook.ipynb) · [Siguiente →](02-lab-sesion-spark.ipynb)

Si vienes de Pandas, este notebook es el puente. Vamos a coger **las mismas cinco filas** y tratarlas primero como lo harías en un script de analista (Pandas) y después como lo hace Spark.

No hace falta que memorices la API. Fíjate en *cuándo* aparece el resultado: en Pandas, en la línea de debajo; en Spark, solo cuando pides una acción (`show`, `count`).

Después creas el lab en `notebooks/trabajo/`. Guion: `02-lab-sesion-spark.ipynb`.

Ejecuta las celdas **aquí**, en este mismo fichero (clase, juntos). Va **montado**: explicación + código + lo que tienes que ver. Lo que construyes tú está en el **lab**.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m01')
print(spark.version, spark.sparkContext.master)


## Un dataset pequeño en Pandas

Vamos a inventar cinco pedidos. En Pandas, `DataFrame(...)` **construye la tabla en ese momento**: las filas ya están en la RAM de este proceso Python. Al ejecutar la celda verás la tabla con una columna extra a la izquierda (`0, 1, 2…`): es el **índice**. Spark no tiene índice de filas; es la primera diferencia que vas a notar.


In [ ]:
import pandas as pd

# Cinco pedidos de juguete. Cada dict es una fila.
pedidos_pd = pd.DataFrame(
    [
        {"order_id": "O1", "status": "paid", "amount": 10.0},
        {"order_id": "O2", "status": "cancelled", "amount": 20.0},
        {"order_id": "O3", "status": "paid", "amount": 5.0},
        {"order_id": "O4", "status": "paid", "amount": 15.0},
        {"order_id": "O5", "status": "pending", "amount": 8.0},
    ]
)
# En Jupyter, el último valor se pinta: ya es la tabla, no un "plan".
pedidos_pd


`pedidos_pd` **es** la tabla. Si la dejas como última expresión, ves filas. No has llamado a nada parecido a `show()`.

La siguiente celda mira la forma y los tipos. `shape` y `dtypes` no lanzan ningún “job”: Pandas ya tiene los datos. Vas a ver `(5, 3)` y que `amount` es numérico. El índice será `[0, 1, 2, 3, 4]`.


In [ ]:
print("shape (filas, columnas):", pedidos_pd.shape)
print("tipos que Pandas adivinó:")
print(pedidos_pd.dtypes)
print("índice (Spark no tiene esto):", list(pedidos_pd.index))
# Estadísticos solo de las columnas numéricas
pedidos_pd.describe()


## Filtrar y agregar en Pandas

`pedidos_pd[condición]` recorre las filas **ahora** y te devuelve **otro** DataFrame, ya recortado. `len(...)` y `.sum()` son números inmediatos.

Al ejecutar: tres filas `paid`, suma de `amount` = `10 + 5 + 15` → **30**.


In [ ]:
# La máscara es una serie de True/False; el [] recorta en el acto.
paid_pd = pedidos_pd[pedidos_pd["status"] == "paid"]
print("tipo de paid_pd:", type(paid_pd))
print("len (filas paid):", len(paid_pd))  # 3, ya calculado
print("suma amount paid:", paid_pd["amount"].sum())  # 30.0
paid_pd


Una columna nueva se asigna con `df["col"] = ...` y **ya está** en el objeto. `groupby` aplasta filas (una por `status`) y el resultado también es inmediato: lo ves al ejecutar, sin `show()`.


In [ ]:
pedidos_pd = pedidos_pd.copy()  # por si reejecutas la celda
pedidos_pd["channel"] = "web"  # asignación eager: la columna existe ya
print(pedidos_pd[["order_id", "channel"]])

# count y suma de amount por estado; Pandas calcula al llegar aquí
pedidos_pd.groupby("status")["amount"].agg(["count", "sum"])


## El mismo dataset en PySpark

Mismas cinco filas, otra forma de pensar. `createDataFrame` no “guarda un Excel en Spark”. Guarda un **plan**: “si alguien pide estas filas, constrúyelas así”.

Al ejecutar vas a ver tres cosas distintas:

1. `print(pedidos_sp)` — un objeto (`DataFrame[order_id: string, …]`), **no** la tabla.
2. `printSchema()` — nombres y tipos. Aquí sí, porque los hemos creado en memoria y Spark los conoce.
3. `show()` — **ahora** sí pinta las cinco filas. `show` es una **acción**: obliga a ejecutar el plan.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, lit, sum as fsum

# Row es una fila con nombre de campo. createDataFrame no "imprime" nada.
pedidos_sp = spark.createDataFrame(
    [
        Row(order_id="O1", status="paid", amount=10.0),
        Row(order_id="O2", status="cancelled", amount=20.0),
        Row(order_id="O3", status="paid", amount=5.0),
        Row(order_id="O4", status="paid", amount=15.0),
        Row(order_id="O5", status="pending", amount=8.0),
    ]
)
print("tipo:", type(pedidos_sp))
print("imprimir el objeto NO es la tabla (compara con pedidos_pd):")
print(pedidos_sp)
pedidos_sp.printSchema()  # contrato de columnas
pedidos_sp.show()  # acción: aquí aparecen las 5 filas


## Dónde se parecen y dónde no

Misma pregunta de negocio (“pedidos cobrados”), dos tiempos distintos.

| Qué haces | Pandas | PySpark |
|-----------|--------|---------|
| Ver las filas | el propio `df` / `head()` | `show()` (acción) |
| Número de filas | `len(df)` / `df.shape[0]` | `count()` (acción) |
| Tipos | `dtypes` | `printSchema()` |
| Índice 0,1,2… | sí | **no** |
| Filtrar | `df[df.col == x]` (ya calculado) | `filter` / `where` (solo alarga el plan) |
| Columna nueva | `df["c"] = ...` | `withColumn` (plan; hay que reasignar) |
| Agrupar | `groupby` (ya calculado) | `groupBy` + `agg` (plan hasta `show`) |
| Dónde viven | RAM de este proceso | particiones (aquí: cores del Codespace) |

La siguiente celda hace el `filter` de `paid`. El primer `print` **no** será una tabla de 3 filas. El `count()` sí dirá **3**, y entonces `show()` las pintará.


In [ ]:
# filter = transformación: Spark anota "más adelante, quédate con paid"
paid_sp = pedidos_sp.filter(col("status") == "paid")
print("después del filter, ¿es una tabla?")
print(paid_sp)  # objeto / plan, no 3 filas
print("count (ahora sí calcula):", paid_sp.count())  # acción → 3
paid_sp.show()


Misma agregación que en Pandas (suma de `amount` por `status`). Encadenamos `groupBy` + `agg` y solo al final `show()`. Si quitaras el `show()`, la celda no pintaría el cuadro: el plan se quedaría quieto.


In [ ]:
(
    pedidos_sp.groupBy("status")
    .agg(fsum("amount").alias("amount_sum"))
    .show()  # sin esto no ves el resultado
)


Columna nueva: en Spark **no** haces `df["channel"] = "web"` (eso pisa o falla; es el gesto de Pandas). Encadenas `withColumn` y **reasignas** (`pedidos_sp = ...`). `lit("web")` es “un literal igual en todas las filas”.


In [ ]:
# withColumn no muta: si no reasignas, pedidos_sp sigue sin channel
pedidos_sp = pedidos_sp.withColumn("channel", lit("web"))
pedidos_sp.select("order_id", "channel").show()


## El puente (y la trampa)

`toPandas()` copia **todas** las filas que le pidas al proceso Python. Con cinco pedidos no pasa nada. Con el fact de NovaShop (miles de líneas) o con un cluster (millones) te comes la RAM del Codespace.

La regla del curso: si quieres mirar en Pandas, primero `limit(...)`. Vas a ver un `DataFrame` de Pandas de **3** filas.


In [ ]:
# limit recorta el plan; toPandas materializa solo esas 3
muestra = pedidos_sp.limit(3).toPandas()
print(type(muestra))
muestra


## Transformación frente a acción

- **Transformación** (`filter`, `withColumn`, `groupBy`, `select`): alarga el plan. En Spark UI (puerto **4040**) no aparece un job nuevo.
- **Acción** (`show`, `count`, `collect`, `write`, `toPandas`): ejecuta. Aparece un job.

Spark no guarda el DataFrame como un Excel. Guarda un **plan**. Hasta una acción, la cocina está apagada.

**Siguiente:** abre el [lab](02-lab-sesion-spark.ipynb) y **crea tu** notebook.
